# LeakAudit - 05b - Verify the D3 `.raw` file format (BLOCKING CHECK)

**Every D3 result depends on reading these files correctly. Run this before
trusting notebooks 05 and 06.**

The dataset description states that each signal is 30 s and that hydrophone
data are `.raw` at 8000 Hz, which implies 240 000 samples per clip. Our loader
currently reads one clip as 642 920 int16 samples, i.e. 80.36 s. That is
2.68 times the expected length, and the file works out at **5.36 bytes per
expected sample** - a number no binary integer or float type can produce.

A fractional byte-per-sample count is the signature of a **text** file. The
dataset also ships a helper, *Python code to convert RAW acoustic data to
pandas DataFrame.py*, which a plain int16 binary would not require.

This notebook settles the question by inspecting the bytes directly, and then
reproduces the provider's own conversion so the two can be compared.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
from pathlib import Path
import src as la
from src import config as cfg, io_utils

M3 = io_utils.scan_d3()
raws = M3[(M3.ext == ".raw") & (~M3.is_bg)].reset_index(drop=True)
print(f"non-background .raw files: {len(raws)}")
probe = Path(raws.path.iloc[0])
size = probe.stat().st_size
print(f"probe file : {probe.name}")
print(f"size       : {size:,} bytes")

EXPECTED_SEC, SR = 30.0, 8000
expected_samples = int(EXPECTED_SEC * SR)
print(f"\nexpected samples for a {EXPECTED_SEC:.0f} s clip at {SR} Hz: {expected_samples:,}")
print(f"bytes per expected sample: {size / expected_samples:.3f}")
print("\n(An integer or float dtype gives 1, 2, 4 or 8. Anything else means the")
print(" file is not a plain binary dump of one channel.)")

non-background .raw files: 120
probe file : BR_CC_0.18 LPS_N_H1.raw
size       : 1,285,840 bytes

expected samples for a 30 s clip at 8000 Hz: 240,000
bytes per expected sample: 5.358

(An integer or float dtype gives 1, 2, 4 or 8. Anything else means the
 file is not a plain binary dump of one channel.)


## 1. Look at the raw bytes

If the file is text, the first bytes are printable digits, signs and newlines.

In [3]:
head = probe.open("rb").read(200)
print("first 200 bytes, raw:")
print(head)
print("\nfirst 200 bytes, decoded as ASCII (errors shown as dots):")
print("".join(chr(b) if 32 <= b < 127 else ("\\n" if b == 10 else ".") for b in head))

printable = sum(1 for b in head if 32 <= b < 127 or b in (9, 10, 13))
print(f"\nprintable fraction of the first 200 bytes: {printable/len(head):.1%}")
IS_TEXT = printable / len(head) > 0.95
print("VERDICT:", "TEXT (ASCII numbers)" if IS_TEXT else "BINARY")

first 200 bytes, raw:
b'\x00\x00F\xdb\x00\x00`\xdb\x00\x00p\xdb\x00\x00r\xdb\x00\x00r\xdb\x00\x00{\xdb\x00\x00\x84\xdb\x00\x00\x88\xdb\x00\x00\x82\xdb\x00\x00y\xdb\x00\x00w\xdb\x00\x00~\xdb\x00\x00\x8b\xdb\x00\x00\x8c\xdb\x00\x00\x8a\xdb\x00\x00\x93\xdb\x00\x00\xa9\xdb\x00\x00\xc3\xdb\x00\x00\xd8\xdb\x00\x00\xeb\xdb\x00\x00\x01\xdc\x00\x00"\xdc\x00\x00B\xdc\x00\x00\\\xdc\x00\x00{\xdc\x00\x00\xac\xdc\x00\x00\xeb\xdc\x00\x00.\xdd\x00\x00g\xdd\x00\x00\x9b\xdd\x00\x00\xda\xdd\x00\x00"\xde\x00\x00a\xde\x00\x00\x96\xde\x00\x00\xc6\xde\x00\x00\xf8\xde\x00\x00+\xdf\x00\x00[\xdf\x00\x00\x87\xdf\x00\x00\xa7\xdf\x00\x00\xca\xdf\x00\x00\xe9\xdf\x00\x00\x0c\xe0\x00\x00 \xe0\x00\x00\x1a\xe0\x00\x00\x14\xe0\x00\x00\x12\xe0\x00\x00!\xe0\x00\x00\x1e\xe0\x00\x00\x0e\xe0'

first 200 bytes, decoded as ASCII (errors shown as dots):
..F...`...p...r...r...{...............y...w...~......................................."...B...\...{...............g..........."...a...............+...[....................... ..

## 2. If it is text, parse it as text

Read it the way the provider intends and check the sample count against the
30 s specification.

In [4]:
if IS_TEXT:
    txt = probe.read_text(errors="ignore")
    # Values may be separated by newlines, commas, tabs or spaces.
    import re
    tokens = [t for t in re.split(r"[\s,;]+", txt.strip()) if t]
    print(f"tokens parsed        : {len(tokens):,}")
    print(f"first 8 tokens       : {tokens[:8]}")
    try:
        vals = np.asarray(tokens, dtype=np.float64)
        ok = True
    except ValueError as e:
        print("could not parse every token as a number:", str(e)[:120])
        vals, ok = None, False
    if ok:
        print(f"parsed samples       : {len(vals):,}")
        print(f"duration at {SR} Hz   : {len(vals)/SR:.2f} s")
        print(f"value range          : [{vals.min():.4g}, {vals.max():.4g}]")
        print(f"mean / std           : {vals.mean():.4g} / {vals.std():.4g}")
        match = abs(len(vals)/SR - EXPECTED_SEC) < 1.0
        print("\nmatches the 30 s specification:", match)
        if not match:
            per_channel = len(vals) / expected_samples
            print(f"samples / expected   : {per_channel:.3f}",
                  "  <- try this many interleaved channels" if abs(per_channel - round(per_channel)) < 0.02 else "")
else:
    print("File is binary; skip to section 3.")

File is binary; skip to section 3.


## 3. Compare against the provider's own converter

The dataset ships a conversion script. Whatever it produces is the reference.
We locate it, show it, and - if it exposes a callable - run it on the probe
file so the two readings can be compared sample for sample.

In [5]:
conv = [p for p in Path(cfg.D3).rglob("*.py") if "convert" in p.name.lower()]
if conv:
    src_txt = conv[0].read_text(errors="ignore")
    print(f"found: {conv[0].name}  ({len(src_txt)} chars)\n")
    print(src_txt[:2000])
else:
    print("No conversion script found under", cfg.D3)
    print("Re-download the dataset; the script is part of the deposit.")

found: Python code to convert RAW acoustic data to pandas DataFrame.py  (1722 chars)

#!/usr/bin/env python
# coding: utf-8

# In[ ]:


import os
import pandas as pd
import soundfile as sf
"""
class for reading RAW acoustic reads, whihc returns a time series as two 
datetime and value arrays.
"""
class RAWSource(object):
    def __init__(self, **kwargs):
        super(RAWSource, self).__init__(**kwargs)
    def raw_to_time_series(self, raw_file_path, channels=1, samplerate=8000, 
                           subtype='PCM_32', endian='LITTLE'):
        signal_raw = sf.read(raw_file_path, channels=channels, 
                             samplerate=samplerate, subtype=subtype, 
                             endian=endian)
        signal = signal_raw[0]
        return signal
"""
Function to convert the RAWSource outputs to a pandas Dataframe.
""" 
def read_raw_to_signals(directory):
    raw_reader = RAWSource()
    with os.scandir(directory) as entries:
        ID = []
        S = []
        

## 4. Decide

Fill in the verdict below. It determines whether the D3 experiments in
notebooks 05 and 06 stand as they are, or must be re-run.

In [6]:
verdict = {
    "file": probe.name,
    "size_bytes": int(size),
    "bytes_per_expected_sample": round(size / expected_samples, 3),
    "looks_like_text": bool(IS_TEXT),
}
if IS_TEXT and 'vals' in dir() and vals is not None:
    verdict["parsed_samples"] = int(len(vals))
    verdict["parsed_duration_s"] = round(len(vals) / SR, 2)
    verdict["matches_30s_spec"] = bool(abs(len(vals)/SR - EXPECTED_SEC) < 1.0)

la.save_json(verdict, "D3_raw_format_check.json")
for k, v in verdict.items():
    print(f"  {k:28s} {v}")

# Resolved: the provider's script specifies PCM_32 little-endian mono at
# 8 kHz, truncated to 240 000 samples. Confirm our reader reproduces that.
x_ok, sr_ok = io_utils.load_d3_raw(probe)
verdict["reader_samples"] = int(len(x_ok))
verdict["reader_duration_s"] = round(len(x_ok) / sr_ok, 2)
verdict["reader_matches_30s"] = bool(abs(len(x_ok) / sr_ok - 30.0) < 1e-6)
print(f"\ncorrected reader: {len(x_ok):,} samples = {len(x_ok)/sr_ok:.2f} s "
      f"-> {'OK' if verdict['reader_matches_30s'] else 'MISMATCH'}")

print("\n" + "=" * 68)
if verdict.get("looks_like_text"):
    print("ACTION REQUIRED")
    print("  The .raw files are text. Reading them with np.fromfile(int16)")
    print("  reinterprets ASCII characters as integers, so every D3 waveform")
    print("  used so far is an artefact of the encoding rather than sound.")
    print("  Fix io_utils.load_d3_signal to parse the text, clear the D3")
    print("  caches in data/, and re-run notebooks 05 and 06.")
    print()
    print("  Note this would also explain the E5 pattern: segments of one file")
    print("  share the same encoding artefact (98.65% under segment-random),")
    print("  while condition-independent evaluation sits at the baseline.")
elif verdict.get("reader_matches_30s"):
    print("RESOLVED - the file is binary PCM_32, not text.")
    print("  The provider's script reads it as 32-bit little-endian PCM, mono,")
    print("  8 kHz, then keeps the first 240 000 samples (30 s).")
    print("  Reading the same bytes as int16 splits each 32-bit word in two,")
    print("  producing a sequence twice as long with a zero in every other")
    print("  position. That halves every frequency and injects a large")
    print("  artificial component near Nyquist, so any spectral feature")
    print("  computed that way is invalid.")
    print()
    print("  io_utils.load_d3_raw now follows the provider's specification and")
    print("  reproduces the 30 s clip exactly. If the D3 caches in data/ were")
    print("  built before this fix, delete data/d3_*.npz and re-run 05 and 06.")
else:
    print("The file is binary but the corrected reader did not return 30 s.")
    print("Compare against the converter output in section 3 before proceeding.")
print("=" * 68)

  file                         BR_CC_0.18 LPS_N_H1.raw
  size_bytes                   1285840
  bytes_per_expected_sample    5.358
  looks_like_text              False

corrected reader: 240,000 samples = 30.00 s -> OK

RESOLVED - the file is binary PCM_32, not text.
  The provider's script reads it as 32-bit little-endian PCM, mono,
  8 kHz, then keeps the first 240 000 samples (30 s).
  Reading the same bytes as int16 splits each 32-bit word in two,
  producing a sequence twice as long with a zero in every other
  position. That halves every frequency and injects a large
  artificial component near Nyquist, so any spectral feature
  computed that way is invalid.

  io_utils.load_d3_raw now follows the provider's specification and
  reproduces the 30 s clip exactly. If the D3 caches in data/ were
  built before this fix, delete data/d3_*.npz and re-run 05 and 06.
